In [ ]:
from datasets import DatasetDict

# HuggingFace split name -> attribute prefix on Splits
SPLIT_NAMES = {"train": "X_train", "validation": "X_val", "test": "X_test"}

# The completion the model learns to produce. Read from ``games_enriched`` — the
# untouched source frame — rather than from the feature frame or metadata:
# pts_diff is an intermediate column deliberately kept out of both, and sourcing
# it here means the LLM needs no change to the shared experiment config.
TARGET_COLUMN = "pts_diff"

if experiment_config is None:
    experiment_config = load_experiment_config(llm_config.data.source_experiment_config)

splits = build_splits(experiment_config)

prompt_config = None
if llm_config.data.serialization_format == "per_team":
    prompt_config = load_prompt_config(llm_config.data.prompt_config)

dataset = DatasetDict(
    {
        name: build_split_dataset(splits, name, llm_config.data.serialization_format, prompt_config)
        for name in SPLIT_NAMES
    }
)

In [ ]:
from datasets import Dataset

features = getattr(splits, SPLIT_NAMES[split_name])
metadata = splits.metadata.reindex(features.index)

if TARGET_COLUMN not in splits.games_enriched.columns:
    raise ValueError(
        f"'{TARGET_COLUMN}' not found in the source feature table, so the LLM "
        "has no completion to learn. Rebuild features with `make build-features`."
    )
target = splits.games_enriched.loc[features.index, TARGET_COLUMN]

In [ ]:
prompts = serialize_frame(
    features,
    metadata,
    fmt=serialization_format,
    raw=splits.games_enriched,
    prompt_config=prompt_config,
)